# 01 — Dataset Exploration

Use this notebook for interactive exploration after running `inspect_dataset.py`.

Goals:
- Understand SQLite table contents
- Visualise speed / vehicle count distributions
- Sanity-check labeling thresholds before running `generate_labels.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.settings import CFG, PROJECT_ROOT, get_path

# Connect to database
raw_dir = get_path('raw_data')
db_candidates = list(raw_dir.glob('*.sqlite')) + list(raw_dir.glob('*.db'))
print('Found:', db_candidates)
con = sqlite3.connect(db_candidates[0])
print('Connected to:', db_candidates[0].name)

In [ ]:
# List all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", con)
print(tables)

In [ ]:
# Inspect TRACKS
tracks = pd.read_sql('SELECT * FROM TRACKS LIMIT 100', con)
tracks.head()

In [ ]:
# Vehicle type distribution
type_col = [c for c in tracks.columns if c.lower() in ['type','class','agent_type','road_user_type']]
if type_col:
    all_tracks = pd.read_sql('SELECT * FROM TRACKS', con)
    all_tracks[type_col[0]].value_counts().plot(kind='bar', title='Road user types')
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect TRAJECTORIES — speed distribution
traj_sample = pd.read_sql('SELECT * FROM TRAJECTORIES LIMIT 5000', con)
traj_sample.head()

In [ ]:
# Plot speed distribution (adjust column name as needed)
speed_col = next((c for c in traj_sample.columns if 'speed' in c.lower() or 'velocity' in c.lower()), None)
if speed_col:
    plt.figure(figsize=(8,4))
    sns.histplot(traj_sample[speed_col].abs(), bins=50, kde=True)
    plt.title(f'Speed distribution (sample) — {speed_col}')
    plt.xlabel('Speed')
    plt.tight_layout()
    plt.show()
else:
    print('No speed column found. Columns:', list(traj_sample.columns))

In [ ]:
# HIGH_LEVEL_ACTIONS — action type distribution
try:
    hla = pd.read_sql('SELECT * FROM HIGH_LEVEL_ACTIONS LIMIT 1000', con)
    action_col = next((c for c in hla.columns if 'action' in c.lower() or 'type' in c.lower()), None)
    if action_col:
        hla[action_col].value_counts().plot(kind='barh', title='High-level actions')
        plt.tight_layout()
        plt.show()
    else:
        print('Columns:', list(hla.columns))
except Exception as e:
    print(f'HIGH_LEVEL_ACTIONS not available: {e}')

In [ ]:
con.close()
print('Done.')